# HelioGuard Phase 6: SFT warm start (Colab T4)

Run the cells in order. The training cell is also the **Resume** cell: run it again after a disconnect and training continues from the last checkpoint on Drive.

Registered run (docs/phase6_gates.md, Amendment 3): Qwen2.5-1.5B-Instruct, trainer `peft`, seed 0, 2 epochs (200 steps), final checkpoint evaluated.

Setup once: Colab secret `GITHUB_TOKEN` (only needed if the repo is private) and a T4 GPU runtime. Nothing secret is stored in this notebook.

In [ ]:
# 1. Fresh clone of the repo (token from the Colab secret, never printed or kept in the remote URL)
import os, subprocess
from google.colab import userdata
REPO = "https://github.com/nithinraj49/helioguard"
!rm -rf /content/helioguard
try:
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None   # public repo: clone without a token
url = REPO.replace("https://", f"https://x-access-token:{token}@") if token else REPO
subprocess.run(["git", "clone", "-q", url, "/content/helioguard"], check=True)
subprocess.run(["git", "-C", "/content/helioguard", "remote", "set-url", "origin", REPO + ".git"], check=True)
%cd /content/helioguard
!git log --oneline -1

In [ ]:
# 2. Install. The pins are the versions used for the registered run: edit them here (and only here) if your
#    Colab needs different ones. torchao>=0.16.0 was needed for the T4 run.
!pip -q install -e ".[sft]" ortools
!pip -q install "peft==0.21.2" "transformers==5.20.2" "accelerate==1.15.0" "torchao>=0.16.0"

In [ ]:
# 3. Check: versions in use (copy these into the run notes) and the GPU
import importlib.metadata as md, torch
for pkg in ("torch", "transformers", "peft", "accelerate", "torchao", "safetensors"):
    print(f"{pkg:13s} {md.version(pkg)}")
print("cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

In [ ]:
# 4. Mount Drive: checkpoints, traces and the final adapter live here, so they survive disconnects
from google.colab import drive
drive.mount("/content/drive")
CKPT = "/content/drive/MyDrive/helioguard/sft_1p5b/checkpoints"
OUT = "/content/drive/MyDrive/helioguard/sft_1p5b"
os.makedirs(CKPT, exist_ok=True)

In [ ]:
# 5. SFT traces: copy from Drive if present, else regenerate (deterministic, CPU, train tasks only) and save to Drive
import os, shutil
TRACES = "/content/helioguard/data/sft/train_traces.jsonl"
DRIVE_TRACES = "/content/drive/MyDrive/helioguard/sft_1p5b/train_traces.jsonl"
os.makedirs(os.path.dirname(TRACES), exist_ok=True)
if os.path.exists(DRIVE_TRACES):
    shutil.copy(DRIVE_TRACES, TRACES)
    print("copied traces from Drive")
else:
    !python scripts/generate_sft_traces.py --workers 2 | tail -25
    shutil.copy(TRACES, DRIVE_TRACES)
    print("generated traces and saved them to Drive")
print(sum(1 for _ in open(TRACES)), "traces (expected 800)")

In [ ]:
# 6. Train / Resume (run this same cell again after a disconnect)
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python scripts/train_sft.py configs/sft_1p5b.json --checkpoint-dir "$CKPT" --out-dir "$OUT"

In [ ]:
# 7. Evaluate the final adapter on the 50 held-out tasks: greedy + temperature 0.7 x 3 seeds, bootstrap CIs, gates (no --greedy-only)
!python -m helioguard.sft_eval --model Qwen/Qwen2.5-1.5B-Instruct --adapter "$OUT/adapter" --out "$OUT/eval" --tag sft_1p5b

Copy `sft_1p5b_eval_summary.json` and `train_summary.json` from Drive back into `results/phase6/` in the repo to commit them. Do not commit checkpoints or adapters unless asked.